# Generate a simulation campaign with a single PredefinedNeuronSet

Sanity-check notebook that builds a `CircuitSimulationScanConfig` containing **exactly one** `PredefinedNeuronSet` (pointing at an existing node set in a local circuit from `examples/data/`), runs the grid-scan generation locally, and asserts the resulting SONATA simulation files have the expected structure.

No entitycore / authentication is involved — generation only writes JSON files to disk; no NEURON simulation is executed.

## 1. Configuration

In [1]:
import shutil
from pathlib import Path

# Local circuit shipped with the repo (in examples/data/)
CIRCUIT_NAME = "N_10__top_nodes_dim6"
CIRCUIT_CONFIG_PATH = (
    Path("../../examples/data/tiny_circuits") / CIRCUIT_NAME / "circuit_config.json"
).resolve()

# Predefined node set name (must exist in the circuit's node_sets.json)
PREDEFINED_NODE_SET = "Layer6"
PREDEFINED_NODE_SET_2 = "Layer5"

# Simulation length [ms]
SIMULATION_LENGTH_MS = 100.0

# Campaign metadata
CAMPAIGN_NAME = "Single PredefinedNeuronSet test"
CAMPAIGN_DESCRIPTION = (
    "Sanity-check campaign: one simulation using a single PredefinedNeuronSet"
)

# Where to write the generated scan (outside the repo, per CLAUDE.md).
# Cleared at the start of each run so the generation step is idempotent.
OUTPUT_ROOT = (
    Path("../../../obi-output/platform_simulation_validation")
    / "generate_campaign_single_predefined_neuron_set"
).resolve()
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## 2. Imports

In [2]:
import json

import obi_one as obi

## 3. Load the circuit and verify the node set exists

In [3]:
assert CIRCUIT_CONFIG_PATH.exists(), f"Circuit config not found at {CIRCUIT_CONFIG_PATH}"

circuit = obi.Circuit(name=CIRCUIT_NAME, path=str(CIRCUIT_CONFIG_PATH))
assert PREDEFINED_NODE_SET in circuit.node_sets, (
    f"Node set '{PREDEFINED_NODE_SET}' not found in circuit '{CIRCUIT_NAME}'. "
    f"Available (first 20): {sorted(circuit.node_sets)[:20]}"
)
print(f"Loaded '{circuit}' with {len(circuit.node_sets)} node sets.")

Loaded 'N_10__top_nodes_dim6' with 147 node sets.


## 4. Build the `CircuitSimulationScanConfig`

Two `PredefinedNeuronSet` blocks are added to the `neuron_sets` dictionary and each receives its own `ConstantCurrentClampSomaticStimulus`, driven by a shared `RegularTimestamps` block. The first neuron set is reused as the simulation target via `initialize.node_set`.

In [4]:
sim_conf = obi.CircuitSimulationScanConfig.empty_config()

sim_conf.set(
    obi.Info(
        campaign_name=CAMPAIGN_NAME,
        campaign_description=CAMPAIGN_DESCRIPTION,
    ),
    name="info",
)

predefined_neuron_set = obi.PredefinedNeuronSet(node_set=[PREDEFINED_NODE_SET])
predefined_neuron_set_2 = obi.PredefinedNeuronSet(node_set=PREDEFINED_NODE_SET_2)
sim_conf.add(predefined_neuron_set, name="PredefinedNeuronSet")
sim_conf.add(predefined_neuron_set_2, name="PredefinedNeuronSet2")

# Shared timestamps (one stimulus onset at t=0)
stim_times = obi.RegularTimestamps(start_time=0.0, number_of_repetitions=1, interval=SIMULATION_LENGTH_MS)
sim_conf.add(stim_times, name="StimTimes")

# One constant-current stimulus per PredefinedNeuronSet
current_stim_1 = obi.ConstantCurrentClampSomaticStimulus(
    timestamps=stim_times.ref,
    neuron_set=predefined_neuron_set.ref,
    duration=SIMULATION_LENGTH_MS,
    amplitude=0.2,
)
current_stim_2 = obi.ConstantCurrentClampSomaticStimulus(
    timestamps=stim_times.ref,
    neuron_set=predefined_neuron_set_2.ref,
    duration=SIMULATION_LENGTH_MS,
    amplitude=0.3,
)
sim_conf.add(current_stim_1, name="CurrentStim_PredefinedNeuronSet")
sim_conf.add(current_stim_2, name="CurrentStim_PredefinedNeuronSet2")

sim_conf.set(
    obi.CircuitSimulationScanConfig.Initialize(
        circuit=circuit,
        node_set=predefined_neuron_set.ref,
        simulation_length=SIMULATION_LENGTH_MS,
    ),
    name="initialize",
)

validated_sim_conf = sim_conf.validated_config()

In [ ]:
EXPECTED_NEURON_SETS = {
    "PredefinedNeuronSet": PREDEFINED_NODE_SET,
    "PredefinedNeuronSet2": PREDEFINED_NODE_SET_2,
}

# Exactly two PredefinedNeuronSets pointing at the expected predefined node sets
# assert set(validated_sim_conf.neuron_sets) == set(EXPECTED_NEURON_SETS), (
#     f"Expected neuron sets {sorted(EXPECTED_NEURON_SETS)}, got {sorted(validated_sim_conf.neuron_sets)}"
# )
# for block_name, predefined_name in EXPECTED_NEURON_SETS.items():
#     ns = validated_sim_conf.neuron_sets[block_name]
#     assert isinstance(ns, obi.PredefinedNeuronSet), (
#         f"Expected PredefinedNeuronSet for '{block_name}', got {type(ns).__name__}"
#     )
#     assert ns.node_set == predefined_name

# # initialize.node_set must reference the first PredefinedNeuronSet
# assert validated_sim_conf.initialize.node_set.block_name == "PredefinedNeuronSet"

# # Exactly one stimulus per neuron set, each pointing at its own PredefinedNeuronSet
# assert set(validated_sim_conf.stimuli) == {
#     "CurrentStim_PredefinedNeuronSet",
#     "CurrentStim_PredefinedNeuronSet2",
# }, f"Unexpected stimuli: {sorted(validated_sim_conf.stimuli)}"
# assert (
#     validated_sim_conf.stimuli["CurrentStim_PredefinedNeuronSet"].neuron_set.block_name
#     == "PredefinedNeuronSet"
# )
# assert (
#     validated_sim_conf.stimuli["CurrentStim_PredefinedNeuronSet2"].neuron_set.block_name
#     == "PredefinedNeuronSet2"
# )

# print(
#     f"Config OK — neuron sets: "
#     f"{ {k: v.node_set for k, v in validated_sim_conf.neuron_sets.items()} }"
# )
# print(f"Stimuli: {sorted(validated_sim_conf.stimuli)}")

AssertionError: 

## 5. Generate the simulation campaign locally

`GridScanGenerationTask.execute()` serializes the scan and its single-coordinate instances. `run_task_for_single_configs` then writes the SONATA `simulation_config.json` and custom `node_sets.json` to disk for each coordinate. Since every parameter is scalar, exactly one coordinate is produced.

In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=validated_sim_conf,
    coordinate_directory_option="ZERO_INDEX",
    output_root=str(OUTPUT_ROOT),
)

# A single PredefinedNeuronSet + scalar initialize params => no parameter sweep
multi_params = grid_scan.multiple_value_parameters(display=True)
assert not multi_params, (
    f"Expected no multi-value parameters, got {multi_params}"
)

grid_scan.coordinate_parameters(display=True)
grid_scan.execute()
obi.run_task_for_single_configs(single_configs=grid_scan.single_configs)

assert len(grid_scan.single_configs) == 1, (
    f"Expected exactly one single-coordinate config, got {len(grid_scan.single_configs)}"
)

[2026-04-20 13:03:57,030] INFO: 
MULTIPLE VALUE PARAMETERS
[2026-04-20 13:03:57,030] INFO: No multiple value parameters found.
[2026-04-20 13:03:57,031] INFO: 
COORDINATE PARAMETERS
[2026-04-20 13:03:57,031] INFO: No coordinate parameters.
[2026-04-20 13:03:57,033] INFO: initialize.circuit is a Circuit instance.
[2026-04-20 13:03:57,048] WARNING: Neuron set empty!


## 6. Inspect the generated SONATA simulation config

Confirms the `PredefinedNeuronSet` survived the round-trip into the SONATA config and that the simulation-level `node_sets.json` resolves the target back to the predefined node set.

In [ ]:
coord_dir = OUTPUT_ROOT / "0"
sim_config_path = coord_dir / "simulation_config.json"
node_sets_path = coord_dir / "node_sets.json"

assert sim_config_path.exists(), f"Missing {sim_config_path}"
assert node_sets_path.exists(), f"Missing {node_sets_path}"

with sim_config_path.open() as fh:
    sonata_cfg = json.load(fh)
with node_sets_path.open() as fh:
    custom_node_sets = json.load(fh)

# Simulation target = first PredefinedNeuronSet
target_node_set = sonata_cfg["node_set"]
print(f"simulation_config.node_set = {target_node_set!r}")
assert target_node_set == "PredefinedNeuronSet"

# Both PredefinedNeuronSets must be materialized with the correct node ids
predefined_blocks = {
    "PredefinedNeuronSet": predefined_neuron_set,
    "PredefinedNeuronSet2": predefined_neuron_set_2,
}
for block_name, block in predefined_blocks.items():
    assert block_name in custom_node_sets, (
        f"Custom node set '{block_name}' not defined in {node_sets_path}"
    )
    generated_ids = set(custom_node_sets[block_name]["node_id"])
    expected_ids = set(
        block.get_neuron_ids(circuit, population=circuit.default_population_name).tolist()
    )
    assert generated_ids == expected_ids, (
        f"'{block_name}' ids {sorted(generated_ids)} do not match "
        f"'{block.node_set}' on the circuit: {sorted(expected_ids)}"
    )
    print(
        f"  '{block_name}' materializes '{block.node_set}' -> "
        f"{len(generated_ids)} node ids."
    )

# Each stimulus must appear in sonata_cfg["inputs"] and target its own PredefinedNeuronSet.
# The generation task suffixes each input with the timestamp index (_0, _1, ...).
expected_stim_targets = {
    "CurrentStim_PredefinedNeuronSet": "PredefinedNeuronSet",
    "CurrentStim_PredefinedNeuronSet2": "PredefinedNeuronSet2",
}
input_keys = set(sonata_cfg["inputs"])
for stim_name, expected_target in expected_stim_targets.items():
    matching = [k for k in input_keys if k.startswith(f"{stim_name}_")]
    assert matching, (
        f"No input starting with '{stim_name}_' in simulation_config.inputs: "
        f"{sorted(input_keys)}"
    )
    for key in matching:
        assert sonata_cfg["inputs"][key]["node_set"] == expected_target, (
            f"Stimulus '{key}' targets "
            f"'{sonata_cfg['inputs'][key]['node_set']}', expected '{expected_target}'"
        )
    print(f"  stimulus '{stim_name}' -> node_set '{expected_target}' ({len(matching)} instance(s))")

simulation_config.node_set = 'PredefinedNeuronSet'
  'PredefinedNeuronSet' materializes 'Layer6' -> 9 node ids.
[2026-04-20 13:03:57,061] WARNING: Neuron set empty!
  'PredefinedNeuronSet2' materializes 'Layer5' -> 0 node ids.
  stimulus 'CurrentStim_PredefinedNeuronSet' -> node_set 'PredefinedNeuronSet' (1 instance(s))
  stimulus 'CurrentStim_PredefinedNeuronSet2' -> node_set 'PredefinedNeuronSet2' (1 instance(s))


## 7. Summary

In [ ]:
print("=== Generation check passed ===")
print(f"Output directory: {OUTPUT_ROOT}")

=== Generation check passed ===
Output directory: /Users/james/Documents/obi/code/obi-main/obi-output/platform_simulation_validation/generate_campaign_single_predefined_neuron_set
